## Import Libraries

In [0]:
import pyspark.sql.functions as F
from pyspark.sql.functions import col, lit, count, countDistinct, avg, sum as _sum, max as _max, min as _min, when, coalesce, row_number, year, month, to_date

## Read all silver tables

In [0]:
df_orders = spark.table("olist.silver.orders")
df_items = spark.table("olist.silver.order_items")
df_products = spark.table("olist.silver.products")
df_customers = spark.table("olist.silver.customers")
df_sellers = spark.table("olist.silver.sellers")
df_payments = spark.table("olist.silver.order_payments")
df_reviews = spark.table("olist.silver.order_reviews")
df_translation = spark.table("olist.silver.product_category_name_translation")

## Add English category names to products table

In [0]:
%sql
-- Show products with their English category name.
-- COALESCE keeps the original name when there is no translation row.

CREATE OR REPLACE TEMP VIEW products_enriched AS
SELECT
    p.*,
    COALESCE(t.product_category_name_english, 'unknown') AS product_category_name_english
FROM olist.silver.products p
LEFT JOIN olist.silver.product_category_name_translation t
    ON p.product_category_name = t.product_category_name

## Aggregate payments per order 

In [0]:
%sql
-- One row per order with payment summary.
CREATE OR REPLACE TEMP VIEW payments_agg AS
SELECT
    order_id,
    ROUND(SUM(payment_value), 2)    AS total_payment_value,
    COUNT(payment_sequential)        AS payment_method_count,
    MAX(payment_installments)        AS max_installments,
    MAX(payment_type)                AS primary_payment_type
FROM olist.silver.order_payments
GROUP BY order_id


## Aggregate reviews per order

In [0]:
%sql
CREATE OR REPLACE TEMP VIEW reviews_agg AS
SELECT
    order_id,
    ROUND(AVG(review_score), 1)     AS avg_review_score,
    MIN(review_score)                AS min_review_score,
    MAX(review_score)                AS max_review_score,
    COUNT(review_id)                 AS review_count,
    MAX(review_creation_date)        AS last_review_date
FROM olist.silver.order_reviews
GROUP BY order_id

##  Build the enriched table

In [0]:
%sql
CREATE OR REPLACE TEMP VIEW orders_enriched AS
SELECT
    -- Order item details
    oi.order_id,
    oi.order_item_id,
    oi.product_id,
    oi.seller_id,
    oi.price,
    oi.freight_value,
    oi.total_item_value,
    oi.freight_ratio,

    -- Order details
    o.customer_id,
    o.order_status,
    o.order_purchase_timestamp,
    o.order_approved_at,
    o.order_delivered_customer_date,
    o.order_estimated_delivery_date,
    o.delivery_time_days,
    o.approval_time_hours,
    o.estimated_vs_actual_days,
    o.is_delivered_late,
    o.order_purchase_date,
    o.order_purchase_year,
    o.order_purchase_month,

    -- Product details
    p.product_category_name,
    p.product_category_name_english,
    p.product_weight_g,

    -- Customer location
    c.customer_city,
    c.customer_state,
    c.customer_zip_code_prefix,

    -- Seller location
    s.seller_city,
    s.seller_state,

    -- Payment summary (pre-aggregated)
    pay.total_payment_value,
    pay.payment_method_count,
    pay.max_installments,
    pay.primary_payment_type,

    -- Review summary (pre-aggregated)
    rev.avg_review_score,
    rev.review_count

FROM olist.silver.order_items oi
INNER JOIN olist.silver.orders o 
    ON oi.order_id = o.order_id
LEFT JOIN products_enriched p 
    ON oi.product_id = p.product_id
LEFT JOIN olist.silver.customers c 
    ON o.customer_id = c.customer_id
LEFT JOIN olist.silver.sellers s 
    ON oi.seller_id = s.seller_id
LEFT JOIN payments_agg pay 
    ON oi.order_id = pay.order_id
LEFT JOIN reviews_agg rev 
    ON oi.order_id = rev.order_id

## Quality Check

In [0]:
df_enriched = spark.table("orders_enriched")

print(f"Total rows: {df_enriched.count()}")
print(f"Expected: ~112,650 (same as order_items)")
print(f"Distinct orders: {df_enriched.select('order_id').distinct().count()}")
print(f"Distinct products: {df_enriched.select('product_id').distinct().count()}")
print(f"Distinct sellers: {df_enriched.select('seller_id').distinct().count()}")
print(f"Null customer_state: {df_enriched.filter(col('customer_state').isNull()).count()}")
print(f"Null avg_review_score: {df_enriched.filter(col('avg_review_score').isNull()).count()}")

df_enriched.printSchema()
df_enriched.display()

## Write to silver

In [0]:
df_enriched.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("olist.silver.orders_enriched")